# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one pseudonymized content page (`content_id`). All metrics are measured over a trailing 90-day window (e.g. `impressions_90d`, `sessions_90d`). Verified below.

In [2]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "https://raw.githubusercontent.com/anshikasri637/machine-learning/main/data/raw/content_refresh_anonymized.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique(), "(should equal row count)")
print("Unique client_id:", df["client_id"].nunique())

Rows: 30000
Unique content_id: 30000 (should equal row count)
Unique client_id: 32


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`, `trend_direction` | Observable performance signals, available before any decision point |
| **Label** | none | Unsupervised — no target to predict |
| **Context (not clustering input)** | `content_type`, `main_intent`, `competition_level` | Used to profile/interpret clusters afterward, not to define them — including them as clustering inputs caused the clusters to just rediscover `content_type` (see finding below) |
| **Excluded** | `search_volume`, `competition`, `cpc` | Structurally 0/null for non-keyword content_types — not a universal performance signal |
| **Excluded** | `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier` | Precalculated buckets of numeric columns already used raw — including both double-weights the same signal |
| **Excluded** | `provider_used`, `model_used` | Content-generation metadata, not a search/engagement signal |

## 3. Verify it with queries (grain, counts, missing values, windows)

In [4]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "https://raw.githubusercontent.com/anshikasri637/machine-learning/main/data/raw/content_refresh_anonymized.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)

# Claim: search_volume/competition/cpc are structurally absent for non-keyword content
sv_null_by_type = df.assign(
    sv_missing=(df["search_volume"].isnull()) | (df["search_volume"] == 0)
).groupby("content_type")["sv_missing"].mean().round(2)
print("Share of rows with search_volume missing/zero, by content_type:")
print(sv_null_by_type)

# Claim: trend_pct null exactly aligns with trend_direction in {flat, new}
trend_null_by_dir = df.groupby("trend_direction")["trend_pct"].apply(lambda s: s.isnull().mean()).round(2)
print("\ntrend_pct null rate by trend_direction:")
print(trend_null_by_dir)

# Claim: avg_position == 0 co-occurs with near-zero impressions (it's a placeholder)
zero_pos = df[df["avg_position"] == 0]
print("\navg_position==0 rows:", len(zero_pos), "| their impressions_90d stats:")
print(zero_pos["impressions_90d"].describe()[["mean","50%","max"]])

Share of rows with search_volume missing/zero, by content_type:
content_type
comparison article    1.0
feedly article        1.0
keyword article       0.4
Name: sv_missing, dtype: float64

trend_pct null rate by trend_direction:
trend_direction
down      0.0
flat      1.0
new       1.0
stable    0.0
up        0.0
Name: trend_pct, dtype: float64

avg_position==0 rows: 1205 | their impressions_90d stats:
mean     1.884647
50%      1.000000
max     36.000000
Name: impressions_90d, dtype: float64


## 4. Data limits

- **Unbalanced history**: different clients have different amounts of tracking history; some pages may look artificially "new" or "flat" simply because their client started tracking recently, not because nothing changed.
- **avg_position == 0 is not a real rank** — it's a placeholder for "too little data to compute a position," confirmed above by its near-zero impressions. Treating it as a genuine top rank would have distorted any position-based archetype.
- **word_count/char_count nulls are partly systematic** — concentrated in `keyword article` content (~28% missing) but not universal, so this looks like an inconsistent tracking gap in one content pipeline rather than random noise. Any cluster this affects should be flagged, not named with full confidence.
- **This is one snapshot in time.** Clustering shows what's true in this 90-day window; it says nothing about causality or what would happen if a page were refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words (observed / directional / decision-support), never causal or 'predicting Google'